# Safestral — authenticated website on a fixed domain

The React frontend, FastAPI API, and Sentinel-Loop pipeline run together at **https://safestral.dpdns.org**, through a Cloudflare **named tunnel**.

**Every route requires login**, including the page, assets, health checks, documentation, uploads, pipeline jobs and WebSocket upgrades. The backend refuses to start without a strong password. Passwords and API keys are never embedded in the frontend.

Before running, add these **Colab Secrets** (key icon in the left sidebar), and enable notebook access:

- `SENTINEL_WEB_PASSWORD`: a unique, randomly generated password with **at least 32 characters**. You choose it and enter it directly in Colab Secrets.
- `CF_TUNNEL_TOKEN`: the connector token for your Cloudflare tunnel `safestral-colab` (just the token, not the whole installation command).

The login name defaults to **`safestral`**; change `WEB_USERNAME` in the configuration cell if needed. The browser asks for your username/password. API clients must authenticate too; knowing the URL grants no access.

One-time Cloudflare setup: register and activate `safestral.dpdns.org`, use the Free DNS plan, and add a Published application route on `safestral-colab`: **Subdomain empty**, **Domain `safestral.dpdns.org`**, **Path empty**, **HTTP service `localhost:8000`**. Domain/tunnel setup is separate from running this notebook. The assigned nameservers prepared in this chat are `eric.ns.cloudflare.com` and `raegan.ns.cloudflare.com`.

Run the cells in order. The first cell closes old cloudflared processes in this runtime, including old public quick tunnels. If an older notebook version is already serving on port 8000, **restart the Colab session once** before starting this version. Data and model artifacts remain in `MyDrive/Colab Notebooks/Mistraton/data/`.

Use a GPU runtime for `finetune-timestral`; other website operations can run on CPU. Runtime lifetimes and availability remain subject to Colab limits. The hostname stays fixed across restarts, while the application is offline when the runtime is stopped.


In [ ]:
# 0) Close old public tunnels before changing security or restarting the backend.
# This affects cloudflared connectors in this Colab runtime only.
import subprocess
closed = subprocess.run(['pkill', '-KILL', '-x', 'cloudflared'], capture_output=True)
if closed.returncode not in (0, 1):
    raise RuntimeError('Could not close old tunnels. Disconnect the old runtime before continuing.')
print('Old cloudflared tunnels in this runtime are closed. No tunnel is opened by this cell.')


In [ ]:
# 1) Mount Drive, load secrets, point the pipeline at Drive-persistent dirs
from google.colab import drive, userdata
import os, sys, shutil, pathlib

drive.mount('/content/drive')  # authorize in the popup; "already mounted" is fine

# Colab exposes the Drive root as /content/drive/MyDrive (NO space — "My Drive" is only the display name)
DRIVE_ROOT = next((p for p in [pathlib.Path('/content/drive/MyDrive'), pathlib.Path('/content/drive/My Drive')] if p.is_dir()), None)
assert DRIVE_ROOT is not None, 'Drive not mounted. Re-run this cell and click the authorization link.'
print('drive root:', DRIVE_ROOT)

# project folder: "Colab Notebooks/Mistraton" (no quotes inside the string; it's MistratoN, not Mistralethon)
PROJECT = DRIVE_ROOT / 'Colab Notebooks' / 'Mistraton'
if not PROJECT.is_dir():
    cn = DRIVE_ROOT / 'Colab Notebooks'
    listing = sorted(p.name for p in cn.iterdir()) if cn.is_dir() else 'NO SUCH FOLDER'
    raise FileNotFoundError(f'{PROJECT} not found. "Colab Notebooks" contains: {listing}')
DRIVE = str(PROJECT)
print('project:', DRIVE)

ENV_FILE = pathlib.Path(DRIVE) / '.env'
assert ENV_FILE.is_file(), f'{ENV_FILE} missing — copy .env.example to .env there and fill in secrets'
for line in ENV_FILE.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        k, _, v = line.partition('=')
        os.environ[k.strip()] = v.strip().strip('"').strip("'")

# Mandatory website login. No password/key is printed or embedded in HTML.
WEB_USERNAME = 'safestral'
PUBLIC_URL = 'https://safestral.dpdns.org'
try:
    web_password = userdata.get('SENTINEL_WEB_PASSWORD')
except Exception:
    raise RuntimeError('Add SENTINEL_WEB_PASSWORD in Colab Secrets and enable notebook access.') from None
if not web_password or len(web_password) < 32:
    raise RuntimeError('SENTINEL_WEB_PASSWORD must be a unique strong password of at least 32 characters.')
os.environ['SENTINEL_WEB_USERNAME'] = WEB_USERNAME
os.environ['SENTINEL_WEB_PASSWORD'] = web_password
os.environ['SENTINEL_PUBLIC_URL'] = PUBLIC_URL
del web_password
if not (PROJECT / 'backend' / 'colab_security.py').is_file():
    raise RuntimeError('Sync the updated backend/colab_security.py and colab_app.py before continuing.')

# persistent data + artifacts on Drive (survive runtime restarts)
os.environ['SENTINEL_DATA_DIR'] = f'{DRIVE}/data'
os.environ['SENTINEL_ARTIFACTS_DIR'] = f'{DRIVE}/data/artifacts'
pathlib.Path(os.environ['SENTINEL_DATA_DIR']).mkdir(parents=True, exist_ok=True)

# NO full repo copy — the Drive FS is slow per file. Import the backend straight from Drive:
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'   # don't litter Drive with __pycache__
sys.path.insert(0, f'{DRIVE}/backend')

# Only the frontend build needs local disk (npm install/build). Copy just its skeleton (~36 small files):
FE = '/content/frontend'
os.makedirs(FE, exist_ok=True)
for item in ['package.json', 'package-lock.json', 'index.html', 'vite.config.js', 'vite.config.ts', 'tsconfig.json']:
    source = PROJECT / 'frontend' / item
    if source.is_file():
        shutil.copy2(source, pathlib.Path(FE) / item)
for folder in ['src', 'public']:
    source = PROJECT / 'frontend' / folder
    if source.is_dir():
        shutil.copytree(source, pathlib.Path(FE) / folder, dirs_exist_ok=True)
os.environ['SENTINEL_DIST_DIR'] = f'{FE}/dist'

print('backend imported from Drive:', f'{DRIVE}/backend')
print('frontend skeleton copied to:', FE)
print('data:', os.environ['SENTINEL_DATA_DIR'])
print('mistral key set:', bool(os.environ.get('MISTRAL_API_KEY')))


In [ ]:
# Security request helpers: no redirects and no credentials in public probes.
import base64, urllib.request, urllib.error

class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None

_probe = urllib.request.build_opener(_NoRedirect())
PROTECTED_PATHS = ['/', '/api/health', '/api/db', '/api/docs', '/api/openapi.json',
                   '/sentinel/health', '/sentinel/data', '/sentinel/pipeline/jobs',
                   '/assets/does-not-exist.js']

def _status(url, *, local_login=False):
    headers = {'Cache-Control': 'no-cache'}
    if local_login:
        if not url.startswith('http://127.0.0.1:8000/'):
            raise RuntimeError('Automated credential checks are restricted to local loopback.')
        login = os.environ['SENTINEL_WEB_USERNAME'] + ':' + os.environ['SENTINEL_WEB_PASSWORD']
        headers['Authorization'] = 'Basic ' + base64.b64encode(login.encode()).decode()
    try:
        with _probe.open(urllib.request.Request(url, headers=headers), timeout=10) as response:
            return response.status
    except urllib.error.HTTPError as error:
        return error.code


In [ ]:
# 2) Backend deps — requirement files read straight from Drive (absolute paths, no cwd assumptions)
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{DRIVE}/backend/requirements.txt'], check=True)
rc = f'{DRIVE}/backend/requirements-colab.txt'
if os.path.exists(rc):
    # GPU fine-tuning deps — only needed for the 'finetune-timestral' pipeline stage
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', rc], check=True)
else:
    print('note:', rc, 'not found — finetune-timestral will need it')
print('deps installed')
# a numba/numpy conflict warning from Colab's preinstalled packages is harmless here


In [ ]:
# 3) Build the React frontend -> /content/frontend/dist (served by the same FastAPI app)
import shutil, subprocess, os
if shutil.which('node') is None:
    subprocess.run('apt-get update -qq && apt-get install -y -qq nodejs npm', shell=True, check=True)
print('node', subprocess.run(['node', '-v'], capture_output=True, text=True).stdout.strip())
os.chdir('/content/frontend')
subprocess.run(['npm', 'install', '--no-audit', '--no-fund'], check=True)
subprocess.run(['npm', 'run', 'build'], check=True)
print('dist ready:', os.environ['SENTINEL_DIST_DIR'])


In [ ]:
# 4) Start a fresh authenticated server on loopback, then verify before tunneling.
import socket, subprocess, time, sys

previous_server = globals().get('_safestral_server')
if previous_server is not None and previous_server.poll() is None:
    previous_server.terminate()
    try:
        previous_server.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_server.kill()
        previous_server.wait(timeout=5)

with socket.socket() as listener_check:
    listener_check.settimeout(1)
    if listener_check.connect_ex(('127.0.0.1', 8000)) == 0:
        raise RuntimeError('An older server is still listening on port 8000. Restart the Colab session and run from the top; do not open a tunnel.')

with open('/content/safestral-backend.log', 'wb') as backend_log:
    _safestral_server = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'colab_app:build_app', '--factory',
         '--host', '127.0.0.1', '--port', '8000', '--log-level', 'warning',
         '--proxy-headers', '--forwarded-allow-ips', '127.0.0.1'],
        cwd=str(PROJECT / 'backend'), env=os.environ.copy(),
        stdout=backend_log, stderr=subprocess.STDOUT,
    )
try:
    for attempt in range(60):
        if _safestral_server.poll() is not None:
            raise RuntimeError('Backend startup failed. Inspect /content/safestral-backend.log.')
        try:
            if _status('http://127.0.0.1:8000/api/health') == 401:
                break
        except (OSError, urllib.error.URLError):
            pass
        time.sleep(0.25)
    else:
        raise RuntimeError('Backend did not become ready; no tunnel will be opened.')
    for path in PROTECTED_PATHS:
        if _status('http://127.0.0.1:8000' + path) != 401:
            raise RuntimeError('Anonymous access was not rejected at ' + path)
    if _status('http://127.0.0.1:8000/api/health', local_login=True) != 200:
        raise RuntimeError('Authenticated health check failed.')
except Exception:
    _safestral_server.kill()
    _safestral_server.wait(timeout=5)
    raise
print('Security checks passed: anonymous requests denied; authenticated local health succeeds.')


In [ ]:
# 5) Reconnect the named tunnel. There is no anonymous quick-tunnel fallback.
import platform, pathlib, subprocess, urllib.request, time

# Re-check local protection immediately before opening any public connector.
for path in PROTECTED_PATHS:
    if _status('http://127.0.0.1:8000' + path) != 401:
        raise RuntimeError('Refusing to open a tunnel: anonymous access at ' + path)
if _status('http://127.0.0.1:8000/api/health', local_login=True) != 200:
    raise RuntimeError('Refusing to open a tunnel: login failed.')

try:
    tunnel_token = userdata.get('CF_TUNNEL_TOKEN').strip()
except Exception:
    raise RuntimeError('Add CF_TUNNEL_TOKEN in Colab Secrets and enable notebook access.') from None
if not tunnel_token or any(c.isspace() for c in tunnel_token):
    raise RuntimeError('CF_TUNNEL_TOKEN must contain only the connector token, not a command.')

connector = pathlib.Path('/content/cloudflared')
if not connector.is_file():
    arch = {'x86_64': 'amd64', 'aarch64': 'arm64'}.get(platform.machine())
    if arch is None:
        raise RuntimeError('Unsupported runtime architecture.')
    download = connector.with_suffix('.download')
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-' + arch, download)
    download.chmod(0o755)
    download.replace(connector)

previous_tunnel = globals().get('_safestral_tunnel')
if previous_tunnel is not None and previous_tunnel.poll() is None:
    previous_tunnel.kill()
    previous_tunnel.wait(timeout=5)
# Pass only connector settings; never pass the website password or model API keys.
connector_env = {k: os.environ[k] for k in ['PATH', 'HOME', 'SSL_CERT_FILE', 'SSL_CERT_DIR', 'TMPDIR'] if k in os.environ}
connector_env['TUNNEL_TOKEN'] = tunnel_token
try:
    with open('/content/safestral-tunnel.log', 'wb') as tunnel_log:
        _safestral_tunnel = subprocess.Popen(
            [str(connector), 'tunnel', '--no-autoupdate', 'run'],
            env=connector_env, stdout=tunnel_log, stderr=subprocess.STDOUT,
        )
finally:
    del tunnel_token, connector_env

try:
    # No password is sent to the public hostname by this notebook.
    for attempt in range(20):
        if _safestral_tunnel.poll() is not None:
            raise RuntimeError('Tunnel exited. Inspect /content/safestral-tunnel.log.')
        try:
            status = _status(PUBLIC_URL + '/api/health')
            if status == 401:
                break
            if status == 200:
                raise RuntimeError('The public hostname permits anonymous access.')
        except (OSError, urllib.error.URLError):
            pass
        time.sleep(1)
    else:
        raise RuntimeError('Domain/tunnel route is not ready or login protection could not be verified.')
    for path in PROTECTED_PATHS:
        if _status(PUBLIC_URL + path) != 401:
            raise RuntimeError('Public authentication check failed at ' + path)
except Exception:
    _safestral_tunnel.kill()
    _safestral_tunnel.wait(timeout=5)
    raise
print('Named tunnel connected; anonymous public requests were rejected.')
print('Open:', PUBLIC_URL)
print('Login name:', WEB_USERNAME, '| password: your SENTINEL_WEB_PASSWORD Colab Secret')


In [ ]:
# 6) Repeat the anonymous-access checks whenever you change a route/configuration.
try:
    for path in PROTECTED_PATHS:
        status = _status(PUBLIC_URL + path)
        print(path, status)
        if status != 401:
            raise RuntimeError('Anonymous request was not rejected: ' + path)
except Exception:
    tunnel = globals().get('_safestral_tunnel')
    if tunnel is not None and tunnel.poll() is None:
        tunnel.kill()
        tunnel.wait(timeout=5)
    raise
print('All listed public paths require authentication.')


## Login and restarting

- Open **https://safestral.dpdns.org** and enter the username (`safestral` by default) and your `SENTINEL_WEB_PASSWORD`. Share credentials only with people you authorize. Browser requests use that login; no API key is published in the HTML or JavaScript.
- Basic login is a shared demo login, not individual user accounts. Use HTTPS for the public address. Cloudflare Access can be added as another authentication layer; its redirects require corresponding changes to the public probe checks.
- API clients must send HTTP Basic authentication on **every** endpoint. CORS and knowledge of the domain name do not grant access. Cross-site browser writes and WebSocket upgrades are rejected.
- The server listens on `127.0.0.1:8000` in the Colab runtime. Cloudflared forwards the fixed hostname to this loopback service.
- Rerun from the top after a runtime restart; reuse the same named tunnel token. The domain stays the same. The website goes offline when Colab stops. Do not rerun an older saved quick-tunnel cell.
- On the first upgrade, run the old-tunnel shutdown cell in the runtime that served the old address, then restart that runtime before starting this version. If you cannot reconnect to the old runtime, terminate that old session through Colab's session manager.
- Drive paths and pipeline/model artifacts stay under `MyDrive/Colab Notebooks/Mistraton`. The frontend source is refreshed before every build so an old credential-bearing bundle is not reused.
- Logs live only in `/content/safestral-backend.log` and `/content/safestral-tunnel.log`; credentials are not printed by the notebook. Review logs before sharing them.
